# 08 — Cross-match Fink diaObjects with Gaia DR3 stars

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Created:** 2026-04-10  

## Objectives

1. Load diaObject positions from Fink alert parquet files (`data_FINK_BLOCK_LC_01/`).
2. Load the Gaia DR3 star catalogue covering all DDFs (`../06_gaia/data_GAIA_STARCAT_DR3_03/allgaia_sources_allddf.csv`).
3. Perform a narrow cone-search cross-match using `astropy.coordinates.match_to_catalog_sky`.
4. Produce a statistical summary of the match (tables + figures).
5. **Key science question:** Are Fink `gaia_star_stable` / `gaia_star_stable_hq` objects also flagged as photometrically stable in Gaia DR3 (i.e., no variability flags)?

## Data paths

| Dataset | Path |
|---------|------|
| Fink src parquet (stable)    | `data_FINK_BLOCK_LC_01/gaia_star_stable_src.parquet`    |
| Fink src parquet (stable HQ) | `data_FINK_BLOCK_LC_01/gaia_star_stable_hq_src.parquet` |
| Gaia DR3 all-DDF catalogue   | `../06_gaia/data_GAIA_STARCAT_DR3_03/allgaia_sources_allddf.csv` |


## 0. Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator
import warnings

warnings.filterwarnings("ignore")

from astropy.coordinates import SkyCoord
import astropy.units as u

# ------------------------------------------------------------------ #
# Paths
# ------------------------------------------------------------------ #
DATA_FINK = "data_FINK_BLOCK_LC_01"
DATA_GAIA = "../06_gaia/data_GAIA_STARCAT_DR3_03"
FIGS_DIR = "figs_FINK_BLOCK_LC_08"

import os

os.makedirs(FIGS_DIR, exist_ok=True)

# ------------------------------------------------------------------ #
# Cross-match tolerance
# ------------------------------------------------------------------ #
MATCH_RADIUS_ARCSEC = 5.0  # narrow cone: 1 arcsec

print(f"Cross-match radius: {MATCH_RADIUS_ARCSEC} arcsec")
print(f"Astropy version: {__import__('astropy').__version__}")

## 1. Load Fink diaObject positions

We load the `src` parquet files for the two Fink groups (`gaia_star_stable` and
`gaia_star_stable_hq`) and deduplicate at the diaObject level (one unique
(ra, dec) per object, taken as the per-object median position).

In [ ]:
# Load both parquet files
df_stable = pd.read_parquet(f"{DATA_FINK}/gaia_star_stable_src.parquet")
df_stable_hq = pd.read_parquet(f"{DATA_FINK}/gaia_star_stable_hq_src.parquet")

print("gaia_star_stable    shape:", df_stable.shape)
print("gaia_star_stable_hq shape:", df_stable_hq.shape)

# Concatenate
df_fink_all = pd.concat([df_stable, df_stable_hq], ignore_index=True)
print("Combined            shape:", df_fink_all.shape)
print("Columns:", df_fink_all.columns.tolist())

In [ ]:
# Build one row per diaObject: median ra, dec; keep group and field
df_obj = df_fink_all.groupby("diaObjectId", as_index=False).agg(
    ra=("r:ra", "median"),
    dec=("r:dec", "median"),
    group=("group", "first"),
    field=("field", "first"),
    n_src=("r:diaSourceId", "count"),
    band_list=("r:band", lambda x: ",".join(sorted(set(x)))),
)

print("Unique diaObjects:", len(df_obj))
print(df_obj["group"].value_counts())
print(df_obj.head(3).to_string())

## 2. Load Gaia DR3 catalogue

In [ ]:
df_gaia = pd.read_csv(f"{DATA_GAIA}/allgaia_sources_allddf.csv")

print("Gaia DR3 catalogue shape:", df_gaia.shape)
print("Columns:", df_gaia.columns.tolist())
print(df_gaia.head(3).to_string())

In [ ]:
# Gaia variability flag columns (True = variable in that class)
VARI_COLS = [
    "in_vari_long_period_variable",
    "in_vari_eclipsing_binary",
    "in_vari_classification_result",
    "in_vari_rrlyrae",
    "in_vari_cepheid",
    "in_vari_planetary_transit",
    "in_vari_short_timescale",
    "in_vari_long_period_variable2",
    "in_vari_eclipsing_binary2",
    "in_vari_rotation_modulation",
    "in_vari_ms_oscillator",
    "in_vari_agn",
    "in_vari_microlensing",
    "in_vari_compact_companion",
]

# Build a synthetic boolean: is_gaia_variable = any variability flag is True
# (NaN treated as False — not flagged in that table)
df_gaia["is_gaia_variable"] = df_gaia[VARI_COLS].fillna(False).astype(bool).any(axis=1)

print(
    "Gaia sources flagged as variable:",
    df_gaia["is_gaia_variable"].sum(),
    f"({100 * df_gaia['is_gaia_variable'].mean():.1f}%)",
)
print("Gaia STABLE column distribution:")
print(df_gaia["STABLE"].value_counts(dropna=False))

## 3. Cone-search cross-match (astropy)

We use `astropy.coordinates.SkyCoord.match_to_catalog_sky` which returns,
for each Fink diaObject, the index of the nearest Gaia source and the
angular separation. We then apply the threshold cut at `MATCH_RADIUS_ARCSEC`.

In [ ]:
# Build SkyCoord objects
coords_fink = SkyCoord(ra=df_obj["ra"].values * u.deg, dec=df_obj["dec"].values * u.deg, frame="icrs")

coords_gaia = SkyCoord(ra=df_gaia["ra"].values * u.deg, dec=df_gaia["dec"].values * u.deg, frame="icrs")

print(f"Fink diaObjects : {len(coords_fink)}")
print(f"Gaia DR3 sources: {len(coords_gaia)}")

In [ ]:
# Match: for each Fink object, find closest Gaia source
idx_gaia, sep2d, _ = coords_fink.match_to_catalog_sky(coords_gaia)
sep_arcsec = sep2d.to(u.arcsec).value

# Apply threshold
matched_mask = sep_arcsec <= MATCH_RADIUS_ARCSEC

print(
    f"Fink objects with Gaia match within {MATCH_RADIUS_ARCSEC}'' : "
    f"{matched_mask.sum()} / {len(df_obj)} "
    f"({100 * matched_mask.mean():.1f}%)"
)

In [ ]:
# Assemble the matched dataframe
df_obj_aug = df_obj.copy()
df_obj_aug["gaia_idx"] = idx_gaia
df_obj_aug["sep_arcsec"] = sep_arcsec
df_obj_aug["has_gaia_match"] = matched_mask

# For matched objects, pull Gaia columns
gaia_cols_to_pull = [
    "source_id",
    "phot_g_mean_mag",
    "astrometric_gof_al",
    "phot_bp_rp_excess_factor",
    "std_dev_mag_g_fov",
    "DDF",
    "STABLE",
    "is_gaia_variable",
] + VARI_COLS

df_gaia_sub = df_gaia[gaia_cols_to_pull].copy()
df_gaia_sub.index = range(len(df_gaia_sub))  # ensure integer index

# Attach Gaia columns at matched rows
df_matched_gaia = df_gaia_sub.iloc[idx_gaia].reset_index(drop=True)
df_matched_gaia.columns = ["gaia_" + c for c in df_matched_gaia.columns]

df_full = pd.concat([df_obj_aug.reset_index(drop=True), df_matched_gaia], axis=1)

# Restrict to matched subset
df_match = df_full[df_full["has_gaia_match"]].copy()
print("Matched dataframe shape:", df_match.shape)
print(df_match.head(3).to_string())

## 4. Statistical summary

In [ ]:
# --- 4.1 Global match statistics ---
n_fink = len(df_obj)
n_match = len(df_match)
n_no_match = n_fink - n_match

print("=" * 55)
print("GLOBAL MATCH STATISTICS")
print("=" * 55)
print(f"  Total Fink diaObjects            : {n_fink}")
print(f"  With Gaia match (<{MATCH_RADIUS_ARCSEC}'')       : {n_match}  ({100 * n_match / n_fink:.1f}%)")
print(f"  Without Gaia match               : {n_no_match}  ({100 * n_no_match / n_fink:.1f}%)")
print()

In [ ]:
# --- 4.2 Match rate per Fink group ---
grp_stats = (
    df_full.groupby("group")["has_gaia_match"]
    .agg(n_total="count", n_matched="sum")
    .assign(match_rate=lambda x: 100 * x["n_matched"] / x["n_total"])
)
print("--- Match rate per Fink group ---")
print(grp_stats.to_string())
print()

In [ ]:
# --- 4.3 Among matched objects: Fink-stable vs Gaia-stable ---
# We check: is the Fink group consistent with Gaia STABLE flag?

# Gaia STABLE column (True/False/NaN)
# Gaia is_gaia_variable (any variability table)

ct_stable = pd.crosstab(df_match["group"], df_match["gaia_STABLE"], margins=True, margins_name="Total")
print("--- Cross-tab: Fink group vs Gaia STABLE flag ---")
print(ct_stable.to_string())
print()

ct_vari = pd.crosstab(
    df_match["group"], df_match["gaia_is_gaia_variable"], margins=True, margins_name="Total"
)
print("--- Cross-tab: Fink group vs Gaia is_variable (any flag) ---")
print(ct_vari.to_string())

In [ ]:
# --- 4.4 Per-DDF match statistics ---
ddf_stats = (
    df_match.groupby("field")
    .agg(
        n_objects=("diaObjectId", "count"),
        gaia_stable=("gaia_STABLE", lambda x: x.sum()),
        gaia_variable=("gaia_is_gaia_variable", lambda x: x.sum()),
        sep_median=("sep_arcsec", "median"),
        sep_max=("sep_arcsec", "max"),
        g_mag_mean=("gaia_phot_g_mean_mag", "mean"),
    )
    .assign(
        frac_gaia_stable=lambda x: 100 * x["gaia_stable"] / x["n_objects"],
        frac_gaia_variable=lambda x: 100 * x["gaia_variable"] / x["n_objects"],
    )
)
print("--- Per-DDF match statistics (matched objects only) ---")
print(ddf_stats.to_string())

In [ ]:
# --- 4.5 Individual variability flags for Fink-stable objects ---
vari_flag_cols_gaia = ["gaia_" + c for c in VARI_COLS]

flag_counts = (
    df_match[vari_flag_cols_gaia]
    .fillna(False)
    .astype(bool)
    .sum()
    .rename(lambda c: c.replace("gaia_in_vari_", ""))
)

print("--- Gaia variability flags triggered among matched Fink objects ---")
print(flag_counts.to_string())
print(
    f"\nObjects with >=1 Gaia vari flag: {df_match['gaia_is_gaia_variable'].sum()} "
    f"/ {len(df_match)} ({100 * df_match['gaia_is_gaia_variable'].mean():.1f}%)"
)

## 5. Figures

In [ ]:
# ================================================================== #
# Fig 1 — Distribution of angular separations for matched objects
# ================================================================== #
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: all separations (including non-matched)
ax = axes[0]
ax.hist(sep_arcsec, bins=100, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(MATCH_RADIUS_ARCSEC, color="red", lw=1.5, ls="--", label=f'match radius = {MATCH_RADIUS_ARCSEC}"')
ax.set_xlabel("Angular separation [arcsec]")
ax.set_ylabel("N diaObjects")
ax.set_title("Nearest Gaia neighbour — all separations")
ax.legend()
ax.xaxis.set_minor_locator(AutoMinorLocator())

# Right: zoom on matched (< match radius)
ax = axes[1]
ax.hist(df_match["sep_arcsec"], bins=50, color="darkorange", edgecolor="none", alpha=0.85)
ax.set_xlabel("Angular separation [arcsec]")
ax.set_ylabel("N diaObjects")
ax.set_title(f'Zoom: matched objects (sep < {MATCH_RADIUS_ARCSEC}")')
ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig01_separation_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig01")

In [ ]:
# ================================================================== #
# Fig 2 — Sky map: matched vs unmatched diaObjects
# ================================================================== #
fig, ax = plt.subplots(figsize=(10, 6))

mask_no = ~df_full["has_gaia_match"]
mask_yes = df_full["has_gaia_match"]

ax.scatter(
    df_full.loc[mask_no, "ra"],
    df_full.loc[mask_no, "dec"],
    s=12,
    c="grey",
    alpha=0.5,
    label="No Gaia match",
    zorder=2,
)
ax.scatter(
    df_full.loc[mask_yes, "ra"],
    df_full.loc[mask_yes, "dec"],
    s=14,
    c="royalblue",
    alpha=0.8,
    label="Gaia matched",
    zorder=3,
)

ax.set_xlabel("R.A. [deg]")
ax.set_ylabel("Dec. [deg]")
ax.set_title("Sky map of Fink diaObjects — Gaia DR3 match status")
ax.legend(markerscale=2)
ax.invert_xaxis()
ax.grid()

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig02_skymap_match_status.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig02")

In [ ]:
# ================================================================== #
# Fig 3 — Sky map: Fink-stable objects coloured by Gaia stability
#   Blue  = STABLE in Gaia (STABLE=True, no vari flag)
#   Red   = variable in Gaia (any vari flag = True)
#   Orange= STABLE=False but no explicit vari flag
# ================================================================== #
def gaia_stability_label(row):
    """Classify matched object by Gaia stability."""
    if row["gaia_is_gaia_variable"]:
        return "Gaia variable"
    elif row["gaia_STABLE"] == True:
        return "Gaia stable"
    else:
        return "Gaia STABLE=False, no vari flag"


df_match = df_match.copy()
df_match["gaia_status"] = df_match.apply(gaia_stability_label, axis=1)

color_map = {
    "Gaia stable": "royalblue",
    "Gaia variable": "crimson",
    "Gaia STABLE=False, no vari flag": "darkorange",
}

fig, ax = plt.subplots(figsize=(11, 6))
for label, color in color_map.items():
    sub = df_match[df_match["gaia_status"] == label]
    if len(sub) == 0:
        continue
    ax.scatter(sub["ra"], sub["dec"], s=16, c=color, alpha=0.8, label=f"{label} (N={len(sub)})", zorder=3)

ax.set_xlabel("R.A. [deg]")
ax.set_ylabel("Dec. [deg]")
ax.set_title("Matched Fink diaObjects — Gaia DR3 stability status")
ax.legend(markerscale=2, fontsize=9)
ax.invert_xaxis()
ax.grid()

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig03_skymap_gaia_stability.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig03")

In [ ]:
# ================================================================== #
# Fig 4 — Stacked bar: Gaia stability fraction per Fink group
# ================================================================== #
ct_status = pd.crosstab(df_match["group"], df_match["gaia_status"])
ct_frac = ct_status.div(ct_status.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Counts
ct_status.plot(
    kind="bar",
    ax=axes[0],
    color=[color_map.get(c, "grey") for c in ct_status.columns],
    edgecolor="k",
    linewidth=0.5,
    rot=20,
)
axes[0].set_title("Gaia stability — counts per Fink group")
axes[0].set_xlabel("Fink group")
axes[0].set_ylabel("N objects")
axes[0].legend(fontsize=8)

# Fractions
ct_frac.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=[color_map.get(c, "grey") for c in ct_frac.columns],
    edgecolor="k",
    linewidth=0.5,
    rot=20,
)
axes[1].set_title("Gaia stability — fraction per Fink group [%]")
axes[1].set_xlabel("Fink group")
axes[1].set_ylabel("Fraction [%]")
axes[1].set_ylim(0, 110)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig04_bar_gaia_stability_per_group.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig04")

In [ ]:
# ================================================================== #
# Fig 5 — Gaia G mag distribution: stable vs variable (matched objects)
# ================================================================== #
fig, ax = plt.subplots(figsize=(8, 5))

for label, color in color_map.items():
    sub = df_match[df_match["gaia_status"] == label]
    if len(sub) == 0:
        continue
    ax.hist(
        sub["gaia_phot_g_mean_mag"].dropna(),
        bins=30,
        histtype="stepfilled",
        alpha=0.55,
        color=color,
        edgecolor=color,
        label=f"{label} (N={len(sub)})",
    )

ax.set_xlabel("Gaia G mean magnitude [mag]")
ax.set_ylabel("N matched objects")
ax.set_title("Gaia G magnitude distribution by stability class")
ax.legend(fontsize=9)
ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig05_Gmag_stability.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig05")

In [ ]:
# ================================================================== #
# Fig 6 — Gaia variability flags breakdown (bar chart)
# ================================================================== #
vari_subset = df_match[df_match["gaia_is_gaia_variable"]]

if len(vari_subset) > 0:
    flag_vals = (
        vari_subset[vari_flag_cols_gaia]
        .fillna(False)
        .astype(bool)
        .sum()
        .rename(lambda c: c.replace("gaia_in_vari_", ""))
        .sort_values(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    flag_vals[flag_vals > 0].plot(kind="bar", ax=ax, color="crimson", edgecolor="k", linewidth=0.5)
    ax.set_title("Gaia DR3 variability flags — Fink-matched objects flagged as variable")
    ax.set_xlabel("Gaia variability table")
    ax.set_ylabel("N objects")
    ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/fig06_gaia_vari_flags.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig06")
else:
    print("No Gaia-variable objects among matched Fink objects — fig06 skipped.")

In [ ]:
# ================================================================== #
# Fig 7 — Angular separation vs Gaia G magnitude
# ================================================================== #
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(
    df_match["gaia_phot_g_mean_mag"],
    df_match["sep_arcsec"],
    c=df_match["gaia_is_gaia_variable"].astype(int),
    cmap="coolwarm",
    s=12,
    alpha=0.6,
    vmin=0,
    vmax=1,
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Gaia variable (0=no, 1=yes)")
cbar.set_ticks([0, 1])

ax.set_xlabel("Gaia G mean magnitude [mag]")
ax.set_ylabel("Match separation [arcsec]")
ax.set_title("Match quality vs Gaia G magnitude")
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig07_sep_vs_Gmag.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig07")

In [ ]:
# ================================================================== #
# Fig 8 — RA/Dec residuals (Fink - Gaia) for matched objects
# ================================================================== #
# Retrieve Gaia (ra, dec) for matched objects
df_match = df_match.copy()
gaia_matched = df_gaia.iloc[df_match["gaia_idx"].values][["ra", "dec"]].values
df_match["gaia_ra"] = gaia_matched[:, 0]
df_match["gaia_dec"] = gaia_matched[:, 1]

# Residuals in arcsec
cos_dec = np.cos(np.radians(df_match["dec"].values))
df_match["dra_arcsec"] = (df_match["ra"] - df_match["gaia_ra"]) * cos_dec * 3600
df_match["ddec_arcsec"] = (df_match["dec"] - df_match["gaia_dec"]) * 3600

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(df_match["dra_arcsec"], df_match["ddec_arcsec"], s=8, c="steelblue", alpha=0.5)
circle = plt.Circle((0, 0), MATCH_RADIUS_ARCSEC, fill=False, color="red", lw=1.5, ls="--")
ax.add_patch(circle)
ax.set_xlabel(r"$\Delta\alpha\,\cos\delta$ [arcsec]")
ax.set_ylabel(r"$\Delta\delta$ [arcsec]")
ax.set_title("Fink − Gaia astrometric residuals")
ax.set_aspect("equal")
ax.axhline(0, color="k", lw=0.7, ls=":")
ax.axvline(0, color="k", lw=0.7, ls=":")

ax = axes[1]
ax.hist(
    df_match["dra_arcsec"], bins=40, histtype="step", color="royalblue", label=r"$\Delta\alpha\cos\delta$"
)
ax.hist(df_match["ddec_arcsec"], bins=40, histtype="step", color="darkorange", label=r"$\Delta\delta$")
ax.set_xlabel("Residual [arcsec]")
ax.set_ylabel("N objects")
ax.set_title("Astrometric residual distributions")
ax.legend()
ax.axvline(0, color="k", lw=0.7, ls=":")

rms_ra = df_match["dra_arcsec"].std()
rms_dec = df_match["ddec_arcsec"].std()
ax.text(
    0.97,
    0.95,
    f'RMS $\\Delta\\alpha\\cos\\delta$ = {rms_ra:.3f}"\nRMS $\\Delta\\delta$ = {rms_dec:.3f}"',
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8),
)

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig08_astrometric_residuals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig08")

## 6. Save matched catalogue

In [ ]:
OUT_DIR = "data_FINK_BLOCK_LC_08"
os.makedirs(OUT_DIR, exist_ok=True)

# Full table (all Fink objects + match info)
df_full.to_csv(f"{OUT_DIR}/fink_diaobj_gaia_match_all.csv", index=False)
print(f"Saved: {OUT_DIR}/fink_diaobj_gaia_match_all.csv  ({len(df_full)} rows)")

# Matched-only table with Gaia columns
df_match.to_csv(f"{OUT_DIR}/fink_diaobj_gaia_match_matched.csv", index=False)
print(f"Saved: {OUT_DIR}/fink_diaobj_gaia_match_matched.csv  ({len(df_match)} rows)")

In [ ]:
df_match

## 7. Summary and conclusions

This notebook performed a narrow cone-search cross-match (radius = 1 arcsec)
between Fink LSST alert diaObjects (groups `gaia_star_stable` and
`gaia_star_stable_hq`) and the Gaia DR3 reference catalogue covering all DDFs.

**Key results:**

- Match rates per group are reported in Section 4.2.
- The `gaia_STABLE` flag and the composite `is_gaia_variable` flag (union of all
  Gaia variability tables) were compared against the Fink stability classification
  (Section 4.3 and Figure 3/4).
- Astrometric residuals (Figure 8) characterise the systematic offset between
  Fink diaObject positions (median over visits) and Gaia DR3 coordinates.
- Objects matched in Fink as `gaia_star_stable_hq` are expected to show the
  highest purity of Gaia-confirmed stable sources.

**Outputs saved:**

| File | Content |
|------|---------|
| `data_FINK_BLOCK_LC_08/fink_diaobj_gaia_match_all.csv`     | All Fink objects + match flag + Gaia columns |
| `data_FINK_BLOCK_LC_08/fink_diaobj_gaia_match_matched.csv` | Matched objects only, full Gaia columns      |
| `figs_FINK_BLOCK_LC_08/fig01_*` … `fig08_*`               | Diagnostic figures                            |
